In [1]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from scipy.stats import ttest_ind
import warnings
warnings.filterwarnings('ignore')

In [2]:
xls = pd.ExcelFile('/content/apms-2014-ch-02-tabs.xls', engine="xlrd")

In [3]:
print(f"\nSheets found in the file ({len(xls.sheet_names)} total):")
for i, name in enumerate(xls.sheet_names):
    print(f"  [{i}] {name}")


Sheets found in the file (15 total):
  [0] Title sheet
  [1] 2.1
  [2] 2.2
  [3] 2.3
  [4] 2.4
  [5] 2.5
  [6] 2.6 
  [7] 2.7
  [8] 2.8
  [9] 2.9 
  [10] 2.10
  [11] 2.11
  [12] 2.12
  [13] Table specifications 
  [14] Corrections


In [4]:
print("\n  → We will use Sheet '2.1' and Sheet '2.3' for our analysis.")


  → We will use Sheet '2.1' and Sheet '2.3' for our analysis.


In [5]:
data = {
    "Age": ["16-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75+"],
    "Male_CMD_%": [10.0, 17.4, 16.3, 13.8, 15.6, 8.1, 5.6],
    "Female_CMD_%": [28.2, 20.7, 22.3, 24.2, 20.2, 14.7, 11.0],
    "Male_N": [249, 355, 468, 489, 541, 538, 418],
    "Female_N": [311, 680, 712, 805, 685, 651, 644]
}
df = pd.DataFrame(data)
df

,Age,Male_CMD_%,Female_CMD_%,Male_N,Female_N
0,16-24,10.0,28.2,249,311
1,25-34,17.4,20.7,355,680
2,35-44,16.3,22.3,468,712
3,45-54,13.8,24.2,489,805
4,55-64,15.6,20.2,541,685
5,65-74,8.1,14.7,538,651
6,75+,5.6,11.0,418,644


In [6]:
df["Male_CMD_Count"] = (df["Male_CMD_%"] / 100) * df["Male_N"]
df["Female_CMD_Count"] = (df["Female_CMD_%"] / 100) * df["Female_N"]
df["Male_CMD_Count"] = df["Male_CMD_Count"].round().astype(int)
df["Female_CMD_Count"] = df["Female_CMD_Count"].round().astype(int)
df["Male_No_CMD"] = df["Male_N"] - df["Male_CMD_Count"]
df["Female_No_CMD"] = df["Female_N"] - df["Female_CMD_Count"]
df

,Age,Male_CMD_%,Female_CMD_%,Male_N,Female_N,Male_CMD_Count,Female_CMD_Count,Male_No_CMD,Female_No_CMD
0,16-24,10.0,28.2,249,311,25,88,224,223
1,25-34,17.4,20.7,355,680,62,141,293,539
2,35-44,16.3,22.3,468,712,76,159,392,553
3,45-54,13.8,24.2,489,805,67,195,422,610
4,55-64,15.6,20.2,541,685,84,138,457,547
5,65-74,8.1,14.7,538,651,44,96,494,555
6,75+,5.6,11.0,418,644,23,71,395,573


In [7]:
male_cmd = df["Male_CMD_Count"].sum()
male_no_cmd = df["Male_No_CMD"].sum()
female_cmd = df["Female_CMD_Count"].sum()
female_no_cmd = df["Female_No_CMD"].sum()
contingency_gender = [
    [male_cmd, male_no_cmd],
    [female_cmd, female_no_cmd]
]
chi2, p, dof, expected = chi2_contingency(contingency_gender)
print("Chi-square (Gender vs CMD):")
print("Chi2:", chi2)
print("p-value:", p)

Chi-square (Gender vs CMD):
Chi2: 69.27513432944679
p-value: 8.564254470717197e-17


In [8]:
df["Total_CMD"] = df["Male_CMD_Count"] + df["Female_CMD_Count"]
df["Total_No_CMD"] = df["Male_No_CMD"] + df["Female_No_CMD"]
contingency_age = df[["Total_CMD", "Total_No_CMD"]].values
chi2_age, p_age, dof_age, expected_age = chi2_contingency(contingency_age)
print("\nChi-square (Age vs CMD):")
print("Chi2:", chi2_age)
print("p-value:", p_age)


Chi-square (Age vs CMD):
Chi2: 100.53555570784262
p-value: 1.9400175267719785e-19


In [9]:
male_percent = df["Male_CMD_%"]
female_percent = df["Female_CMD_%"]
t_stat, p_ttest = ttest_ind(male_percent, female_percent)

print("T-test (Male vs Female CMD %):")
print("t-stat:", t_stat)
print("p-value:", p_ttest)

T-test (Male vs Female CMD %):
t-stat: -2.8099130342890377
p-value: 0.015751952672210624


In [11]:
def format_p(p):
    if p < 0.001:
        return "<0.001"
    else:
        return round(p, 4)

results = pd.DataFrame({
    "Test": ["Chi-square", "Chi-square", "t-test"],

    "Variables": [
        "Gender vs CMD",
        "Age vs CMD",
        "Male vs Female CMD Rates"
    ],

    "Null Hypothesis": [
        "No association between gender and CMD",
        "No association between age and CMD",
        "Mean CMD rate equal in males and females"
    ],

    "p-value": [
        format_p(p),
        format_p(p_age),
        format_p(p_ttest)
    ],
    "Conclusion": [
        "Reject H₀ (Statistically Significant)" if p < 0.05 else "Fail to Reject H₀",
        "Reject H₀ (Statistically Significant)" if p_age < 0.05 else "Fail to Reject H₀",
        "Reject H₀ (Statistically Significant)" if p_ttest < 0.05 else "Fail to Reject H₀"
    ]
})
# Add summary row
summary_text = (
    "Summary: All null hypotheses are rejected. "
    "Gender and age significantly affect CMD prevalence, "
    "and CMD rates differ significantly between males and females."
)
summary_row = pd.DataFrame({
    "Test": ["Summary"],
    "Variables": [""],
    "Null Hypothesis": [""],
    "p-value": [""],
    "Conclusion": [summary_text]
})
results = pd.concat([results, summary_row], ignore_index=True)
results

,Test,Variables,Null Hypothesis,p-value,Conclusion
0,Chi-square,Gender vs CMD,No association between gender and CMD,<0.001,Reject H₀ (Statistically Significant)
1,Chi-square,Age vs CMD,No association between age and CMD,<0.001,Reject H₀ (Statistically Significant)
2,t-test,Male vs Female CMD Rates,Mean CMD rate equal in males and females,0.0158,Reject H₀ (Statistically Significant)
3,Summary,,,,Summary: All null hypotheses are rejected. Gen...


In [12]:
results.to_csv("hypothesis_test_results.csv", index=False)